In [ ]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

import torch

sys.path.append(".")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())


In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))


# MITweet: relevance recognition and ideology detection

Fine-tunes the pretrained checkpoints and the baselines on
[MITweet](https://github.com/LST1836/MITweet) (EMNLP 2023), read from the local checkout at
`../MITweet`. Two tasks:

- **relevance** — one row per tweet, 12 binary facet labels, weighted BCE.
- **ideology** — one row per (tweet, *relevant* facet), 3-class left/center/right. Gold
  relevance is assumed, as in the paper, so this is the pipeline's second stage, not an
  end-to-end evaluation.

Ideology reuses the `bias_*` keys and `bias_head`, because MITweet's `I1`-`I12` are
`0=left / 1=center / 2=right` — exactly the AllSides bias labels.

Two knobs beyond the model:

- `prepend` describes the facet to the model: `indicators` (the log-odds keywords from
  `Indicators.txt`, joined by `</s>` as MITweet does), `facet_name`, or `none` as a control.
- `split` is `random` (the shipped split, every facet in train and test) or `facet` (some
  facets held out and seen only at test).

In [ ]:
import glob
from pathlib import Path

from huggingface_hub import snapshot_download

from config import load_run_config
from finetuning import aggregate as agg
from finetuning.experiments import run_mitweet_experiment, ExperimentConfig
from finetuning.mitweet import (
    FACET_FOLDS, FACET_NAMES, HELD_OUT_FACETS, NUM_FACET_FOLDS,
    MITweetConfig, held_out_for, variant_name,
)
from finetuning.models import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER


In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")


In [ ]:
def find_tlp_checkpoints(config_glob="run_configs/tlp_*.yaml"):
    """Locate the checkpoint each tlp_* pretraining run left behind.
    """
    checkpoints = []
    for config_path in sorted(glob.glob(config_glob)):
        label = Path(config_path).stem
        output_dir = Path(load_run_config(config_path).output_dir)
        epochs = sorted(
            output_dir.glob("epoch-*.pt"),
            key=lambda p: int(p.stem.split("-")[1]),
        )
        if not epochs:
            print(f"  SKIP {label}: no epoch-*.pt under {output_dir}")
            continue
        print(f"  {label}: {epochs[-1]}")
        checkpoints.append((str(epochs[-1]), label))
    return checkpoints


print("tlp checkpoints:")
TLP_CHECKPOINTS = find_tlp_checkpoints()
print(f"\nfound {len(TLP_CHECKPOINTS)} of 4")


In [ ]:
BASELINES = [
    (BERT, "bert"),
    (BART, "bart"),
    (ROBERTA, "roberta"),
    (POLITICS, "politics"),
    (ideology_pt, "ideology"),
]

MITWEET_ROOT = "results_mitweet"


def run_all_models(models, mitweet_config, seed, root=MITWEET_ROOT):
    """Fine-tune each of `models` on one MITweet variant, under one seed.
    """
    # One directory per variant, so `agg.discover_results` reads a single comparison. The
    # facet split rotates its held-out triple, so it gets a fold level above the seed, which
    # keeps `aggregate`'s `seed_N` leaf intact.
    loc = f"{root}/{variant_name(mitweet_config)}"
    if mitweet_config.split == "facet":
        loc = f"{loc}/fold_{mitweet_config.fold}"
    loc = f"{loc}/seed_{seed}"
    os.makedirs(loc, exist_ok=True)
    results = {}
    for model_ref, model_name in models:
        exp = ExperimentConfig(patience=2, num_epochs=10, save_model=False, seed=seed)
        print(f"\n{'='*60}")
        held_out = f"  |  Held out: {held_out_for(mitweet_config)}" if mitweet_config.split == "facet" else ""
        print(f"Model: {model_name}  |  Variant: {variant_name(mitweet_config)}{held_out}  |  Seed: {seed}")
        print('='*60)
        results[model_name] = run_mitweet_experiment(
            model=model_ref,
            loc=loc,
            mitweet_config=mitweet_config,
            experiment_config=exp,
            model_name=model_name,
        )
    return results


In [ ]:
seeds = [42, 1, 13, 1234, 6789]
MODELS = BASELINES + TLP_CHECKPOINTS

PREPEND_MODES = ["indicators", "facet_name", "none"]

print(f"{len(MODELS)} models x {len(seeds)} seeds")
print(f"\nfacet folds ({NUM_FACET_FOLDS}, a disjoint partition of all 12 facets):")
for fold, triple in enumerate(FACET_FOLDS):
    names = ", ".join(FACET_NAMES[k - 1] for k in triple)
    print(f"  fold {fold}: {triple}  {names}")
print(f"\nfold 0 is the original HELD_OUT_FACETS {HELD_OUT_FACETS}, so it reproduces the "
      "runs already on disk.")


## Relevance recognition

One row per tweet, the shipped random split. `prepend` has no meaning here — the model
predicts all 12 facets from the tweet alone — so the config is fixed.

In [ ]:
for seed in seeds:
    print(f"\n{'#'*60}\nrelevance, seed: {seed}\n{'#'*60}")
    run_all_models(
        models=MODELS,
        mitweet_config=MITweetConfig(task="relevance", prepend="none", split="random"),
        seed=seed,
    )


## Ideology detection, random split

Every facet appears in train and test. Three prepend modes, so the effect of describing the
facet is separable from the backbone.

In [ ]:
for prepend in PREPEND_MODES:
    for seed in seeds:
        print(f"\n{'#'*60}\nideology / {prepend} / random, seed: {seed}\n{'#'*60}")
        run_all_models(
            models=MODELS,
            mitweet_config=MITweetConfig(task="ideology", prepend=prepend, split="random"),
            seed=seed,
        )


## Ideology detection, held-out facets

Train and validation see only the facets *not* held out; test sees only those. Model selection
never touches a held-out facet, so the test number is not leaked.

The `none` arm is the floor here: with no description of the facet, a model has no way to
know which of twelve questions it is being asked, so anything above chance in the prepend
arms is the prefix doing work.

**Four folds rotate the held-out triple.** This split's test set is 864 rows, small enough that
a single triple's result is dominated by which facets it happened to hold out -- a paired
bootstrap over the existing runs puts the interval at about +/-8 F1. `FACET_FOLDS` partitions all
12 facets into four triples, each drawing from three different domains. Fold 0 is the original
`(3, 8, 11)`, so it reproduces what is already on disk rather than replacing it.

Results land in `results_mitweet/{variant}/fold_{f}/seed_{s}`. Read one fold with
`agg.discover_results(.../fold_0)` and all four with `agg.discover_folds(...)`.

In [ ]:
for prepend in PREPEND_MODES:
    for fold in range(NUM_FACET_FOLDS):
        for seed in seeds:
            print(f"\n{'#'*60}\nideology / {prepend} / facet fold {fold}, seed: {seed}\n{'#'*60}")
            run_all_models(
                models=MODELS,
                mitweet_config=MITweetConfig(
                    task="ideology", prepend=prepend, split="facet", fold=fold,
                ),
                seed=seed,
            )


## Cross-seed analysis

Everything below reads the per-seed metrics JSONs off disk — no model, no GPU. The runs
above do not need to have happened in this session.

MITweet's metric names, and where they live in these JSONs:

| paper | ideology | relevance |
|---|---|---|
| Micro-F1 | `f1_macro` (pooled, sklearn-macro over 3 classes) | `f1_micro` (every cell flattened) |
| Macro-F1 | `facet_f1_macro` (mean over facets) | `f1_macro` (mean over facets) |
| Micro-Acc | `accuracy` | — |
| Macro-Acc | `facet_accuracy` | — |

In [ ]:
VARIANT = "ideology_indicators_random"

results = agg.discover_results(f"{MITWEET_ROOT}/{VARIANT}")
print(f"{len(results)} runs under {MITWEET_ROOT}/{VARIANT}")

# A model missing seeds gets a mean over fewer runs, and n is the only place that shows up.
agg.coverage(results)


In [ ]:
IDEOLOGY_METRICS = ["f1_macro", "accuracy", "facet_f1_macro", "facet_accuracy"]

agg.summary_table(results, metrics=IDEOLOGY_METRICS)


In [ ]:
# Every variant side by side: does describing the facet help, and does it survive holding
# facets out? One row per (variant, model).
import pandas as pd

rows = []
for variant in sorted(os.listdir(MITWEET_ROOT)):
    runs = agg.discover_results(f"{MITWEET_ROOT}/{variant}")
    if not runs:
        continue
    metrics = ["f1_macro", "f1_micro"] if variant.startswith("relevance") else IDEOLOGY_METRICS
    table = agg.summary_table(runs, metrics=metrics)
    for model, row in table.iterrows():
        rows.append({"variant": variant, "model": model, **row.to_dict()})

pd.DataFrame(rows).set_index(["variant", "model"])


In [ ]:
# Per-facet ideology F1 for one model, so a variant that only helps the frequent facets is
# visible as such. Facet indices here are 0-based; FACET_NAMES is in the same order.
import json

MODEL = "tlp_16"

runs = agg.by_model(agg.discover_results(f"{MITWEET_ROOT}/{VARIANT}"))[MODEL]
rows = []
for run in runs:
    payload = json.load(open(run.path))
    for facet, name in enumerate(FACET_NAMES):
        key = f"f1_facet_{facet}"
        if key in payload:
            rows.append({"facet": name, "seed": run.seed, "f1": payload[key]})

frame = pd.DataFrame(rows)
frame.groupby("facet")["f1"].agg(["mean", "std", "count"]).round(2)


In [ ]:
# Ideology-only read-outs: the ensemble across seeds, how much the seeds disagree, and which
# class confusions dominate. Relevance predictions are 12-vectors, so `align_by_id` raises
# on them rather than reporting a misjoined number.
agg.print_reports(agg.discover_results(f"{MITWEET_ROOT}/{VARIANT}"))
